# Rates derivatives: deposits, FRA, IRS

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb` and `02_pricing/pricing_across_asset_classes.ipynb`. We price **deposits** at two horizons, a **forward rate agreement**, and a vanilla **IRS**.


## Concept

- **Deposits:** PV from quoted rate vs discount factors.
- **FRA:** payoff linked to fixing vs strike forward rate.
- **IRS:** fixed leg vs projected floating leg.

**PV01** / **bucketed DV01** summarize parallel vs tenor-local rate sensitivities when registered.


In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics, usd_ois_curve, usd_sofr_curve
from finstack_quant.core.market_data import MarketContext
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    price_instrument_with_metrics,
    validate_instrument_json,
)

print("Imports OK (rates derivatives).")


Imports OK (rates derivatives).


## Instrument JSON

**IRS fixed leg:** The fixed rate is **~4.5%** to sit near **par** against the bundled OIS/SOFR snapshot; a **3%** coupon would be intentionally off-market (large PV from fixed–float mismatch) if you use it for illustration.


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

def deposit(mid: str, mat: str, rate: float) -> str:
    return json.dumps(
        {
            "type": "deposit",
            "spec": {
                "id": mid,
                "notional": {"amount": 1_000_000.0, "currency": "USD"},
                "start_date": AS_OF_STR,
                "maturity": mat,
                "day_count": "Act360",
                "quote_rate": rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        }
    )

dep_3m = deposit("DEP-3M", "2025-04-15", 0.0505)
dep_1y = deposit("DEP-1Y", "2026-01-15", 0.0490)

fra = json.dumps(
    {
        "type": "forward_rate_agreement",
        "spec": {
            "id": "FRA-3x6",
            "notional": {"amount": "10000000", "currency": "USD"},
            "start_date": "2025-04-15",
            "maturity": "2025-07-15",
            "fixed_rate": "0.048",
            "day_count": "Act360",
            "discount_curve_id": "USD-OIS",
            "forward_curve_id": "USD-SOFR-3M",
            "fixing_date": "2025-04-10",
            "reset_lag": 2,
            "side": "receive",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

irs = json.dumps(
    {
        "type": "interest_rate_swap",
        "spec": {
            "id": "IRS-5Y-USD",
            "notional": {"amount": 10_000_000.0, "currency": "USD"},
            "side": "pay",
            "fixed": {
                "discount_curve_id": "USD-OIS",
                "rate": 0.045,
                "frequency": {"count": 6, "unit": "months"},
                "day_count": "Thirty360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "start": "2025-04-15",
                "end": "2030-04-15",
                "par_method": None,
                "compounding_simple": True,
            },
            "float": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": 0.0,
                "frequency": {"count": 3, "unit": "months"},
                "day_count": "Act360",
                "bdc": "modified_following",
                "calendar_id": None,
                "stub": "None",
                "reset_lag_days": 2,
                "start": "2025-04-15",
                "end": "2030-04-15",
                "compounding": "Simple",
            },
            "attributes": {},
        },
    }
)

for label, js in (("dep_3m", dep_3m), ("dep_1y", dep_1y), ("fra", fra)):
    validate_instrument_json(js)
    print(label, "OK")


dep_3m OK
dep_1y OK
fra OK


## MarketContext


In [3]:
ois = usd_ois_curve(AS_OF)
sofr = usd_sofr_curve(AS_OF)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("market_json chars:", len(market_json))


market_json chars: 1672


## Pricing


In [4]:
for label, js in (("dep_3m", dep_3m), ("dep_1y", dep_1y), ("fra", fra)):
    out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
    print(label, ValuationResult.from_json(out))

out = price_instrument_with_metrics(irs, market_json, AS_OF_STR, model="discounting")
print("irs", ValuationResult.from_json(out))


dep_3m ValuationResult(id="DEP-3M", price=1005134.7925, currency=USD, metrics=0)
dep_1y ValuationResult(id="DEP-1Y", price=1018190.1389, currency=USD, metrics=0)
fra ValuationResult(id="FRA-3x6", price=-7421.5768, currency=USD, metrics=0)
irs ValuationResult(id="IRS-5Y-USD", price=88983.1704, currency=USD, metrics=0)


## Metrics


In [5]:
metrics_req = ["dv01", "bucketed_dv01", "npv01"]
for label, js in (("dep_3m", dep_3m), ("irs", irs)):
    out = price_instrument_with_metrics(
        js, market_json, AS_OF_STR, model="discounting", metrics=metrics_req
    )
    vr = ValuationResult.from_json(out)
    print("—", label)
    print_metrics(vr, metrics_req)
print("PV01 / bucketed DV01: interpret as parallel vs key-rate style sensitivities when present.")


— dep_3m
  dv01: -24.784145570127293
  bucketed_dv01: -24.784145575889852
— irs
  dv01: 4460.509098880322
  bucketed_dv01: 4589.487221554387
PV01 / bucketed DV01: interpret as parallel vs key-rate style sensitivities when present.


## Takeaways

- **Deposits** anchor short-end PV; **FRAs** and **swaps** need consistent **forward** and **discount** IDs.
- **Unseasoned** swaps avoid missing historical fixings before `as_of`.
- Request **`bucketed_dv01`** when you want tenor-bucket narratives; omit silently if not registered.
